# Problem 1: Daily summaries of NYC Taxi data with Dask

Calculate daily summaries of key metrics for 12 months of 2023 NYC TLC yellow taxi data:

- Number of trips
- Average `trip_distance`
- Max `mta_tax`
- 95th percentile of `fare_amount`
- Min `tip_amount`
- Average `passenger_count` (rounded up)

Run this notebook locally first (a `LocalCluster`), then point it at the EC2 cluster.

## 1. Setup

In [ ]:
import os

import dask.dataframe as dd
import numpy as np
import pandas as pd
from dask.distributed import Client, LocalCluster

YEAR = 2023
GROUP_KEYS = ["dt_year", "dt_month", "dt_day"]

# Where your 2023 files live, e.g. s3://your-bucket/your-prefix
S3_PATH = os.getenv("TAXI_S3_PATH", "s3://your-bucket/your-prefix")

# Leave as None to use a local cluster. On the EC2 scheduler node use
# "tcp://<SCHEDULER_PRIVATE_IP>:8786"
SCHEDULER = os.getenv("SCHEDULER_ADDRESS")

In [ ]:
if SCHEDULER:
    client = Client(SCHEDULER)
else:
    client = Client(LocalCluster(n_workers=2, threads_per_worker=2, memory_limit="3GB"))

print(f"Workers: {len(client.scheduler_info()['workers'])}")
print(f"Dashboard: {client.dashboard_link}")
client

## 2. Read the data

Dask reads lazily: nothing is loaded until we call `.compute()` or `len()`.

In [ ]:
files = [f"{S3_PATH.rstrip('/')}/yellow_tripdata_{YEAR}-{m:02d}.parquet" for m in range(1, 13)]
columns = ["tpep_pickup_datetime", "trip_distance", "mta_tax",
           "fare_amount", "tip_amount", "passenger_count"]

nyc_tlc = dd.read_parquet(files, columns=columns, split_row_groups=True)
print(f"{nyc_tlc.npartitions} partitions")
nyc_tlc.head()

## 3. Derive date columns and filter to 2023

In [ ]:
pickup = nyc_tlc["tpep_pickup_datetime"]
nyc_tlc = nyc_tlc.assign(dt_year=pickup.dt.year, dt_month=pickup.dt.month, dt_day=pickup.dt.day)

# The raw files contain a few stray dates outside 2023
nyc_tlc_year = nyc_tlc[nyc_tlc["dt_year"] == YEAR]

# No .persist() here: holding ~38 million rows in worker memory pushes small
# workers past 95% of their memory limit and Dask kills them.

## 4. Daily summaries

In [ ]:
grouped = nyc_tlc_year.groupby(GROUP_KEYS)

daily = grouped.agg(
    num_trips=("trip_distance", "size"),
    mean_trip_distance=("trip_distance", "mean"),
    max_mta_tax=("mta_tax", "max"),
    min_tip_amount=("tip_amount", "min"),
    mean_passenger_count=("passenger_count", "mean"),
).compute()

# Dask has no groupby quantile. groupby().apply(quantile) shuffles every row
# between workers and ran our workers out of memory. Fares are whole cents, so
# instead count trips at each fare per day (a small table) and read the exact
# 95th percentile off the cumulative counts.
def percentile_from_counts(counts, q=0.95):
    counts = counts.sort_values("fare_cents")
    cumulative = counts["n"].to_numpy().cumsum()
    values = counts["fare_cents"].to_numpy()
    position = q * (cumulative[-1] - 1)
    low, high = int(np.floor(position)), int(np.ceil(position))
    value_low = values[np.searchsorted(cumulative, low, side="right")]
    value_high = values[np.searchsorted(cumulative, high, side="right")]
    return (value_low + (value_high - value_low) * (position - low)) / 100

fares = nyc_tlc_year[nyc_tlc_year["fare_amount"].notnull()]
fare_counts = (fares.assign(fare_cents=(fares["fare_amount"] * 100).round().astype("int64"))
               .groupby(GROUP_KEYS + ["fare_cents"]).size()
               .compute().rename("n").reset_index())
q95 = (fare_counts.groupby(GROUP_KEYS)
       .apply(percentile_from_counts, include_groups=False)
       .rename("q95_fare_amount"))

daily = daily.join(q95).reset_index()
daily["mean_passenger_count"] = np.ceil(daily["mean_passenger_count"])
print(f"{int(daily['num_trips'].sum()):,} rows in {YEAR}")

## 5. Sort (descending by date) and save

In [ ]:
daily_averages = (daily[GROUP_KEYS + ["num_trips", "mean_trip_distance", "max_mta_tax",
                                      "q95_fare_amount", "min_tip_amount", "mean_passenger_count"]]
                  .sort_values(GROUP_KEYS, ascending=False)
                  .reset_index(drop=True))

daily_averages.to_csv("daily_averages.csv", index=False)
print(f"{len(daily_averages)} days saved to daily_averages.csv")
daily_averages.head(20)

## 6. Plot trips per day

In [ ]:
dates = pd.to_datetime(daily_averages[GROUP_KEYS].rename(
    columns={"dt_year": "year", "dt_month": "month", "dt_day": "day"}))
(daily_averages.set_index(dates)["num_trips"].sort_index()
 .plot(figsize=(10, 3), title=f"Trips per day, {YEAR}"));

In [ ]:
client.close()